# Notebook 01 — Análisis Exploratorio de Datos (EDA)

## Objetivo

Explorar el dataset de noticias fake y real para entender su estructura, distribución temporal, longitudes de texto y posibles sesgos temáticos.

> **Aclaración metodológica:** Este análisis describe patrones en los datos. No implica que el modelo posterior verifique hechos contra la realidad; solo aprenderá correlaciones lingüísticas del dataset.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

## 1. Carga de datos y etiquetado

In [ ]:

fake_df = pd.read_csv(DATA_RAW / 'Fake.csv')
true_df = pd.read_csv(DATA_RAW / 'True.csv')

fake_df['label'] = 1
true_df['label'] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
print(f'Total registros: {len(df):,}')
df.head()


## 2. Columnas, nulos y duplicados

In [ ]:

print('Columnas:', df.columns.tolist())
print('\nNulos por columna:')
print(df.isnull().sum())
print('\nDuplicados exactos:', df.duplicated().sum())
print('\nDuplicados en title+text:', df.duplicated(subset=['title', 'text']).sum())


## 3. Parsing de fechas

In [ ]:

from nlp.preprocessing import parse_dates, add_text_columns

df = parse_dates(df)
invalid_dates = df['parsed_date'].isna().sum()
print(f'Fechas inválidas tras parsing multi-formato: {invalid_dates}')
print('Formatos detectados: "%B %d, %Y" (ej. May 31, 2017) y "%d-%b-%y" (ej. 19-Feb-18)')
df = df.dropna(subset=['parsed_date']).reset_index(drop=True)
df = add_text_columns(df)
df[['date', 'parsed_date']].head()


## 4. Distribución de clases

In [ ]:

class_counts = df['label'].value_counts().sort_index()
total = len(df)
print('Distribución de clases:')
for label, name in [(0, 'real'), (1, 'fake')]:
    count = class_counts.get(label, 0)
    pct = count / total * 100
    print(f'  {name}: {count:,} ({pct:.2f}%)')


## 5. Distribución temporal

In [ ]:

for label, name in [(0, 'real'), (1, 'fake')]:
    subset = df[df['label'] == label]
    print(f'{name}: {subset["parsed_date"].min().date()} -> {subset["parsed_date"].max().date()}')

df['year_month'] = df['parsed_date'].dt.to_period('M').astype(str)
monthly = df.groupby(['year_month', 'label']).size().unstack(fill_value=0)
monthly.columns = ['real', 'fake']
monthly.tail(10)


## 6. Longitud de textos (palabras)

In [ ]:

for col in ['title', 'text', 'full_text']:
    df[f'{col}_word_count'] = df[col].fillna('').astype(str).str.split().str.len()

length_stats = df.groupby('label')[['title_word_count', 'text_word_count', 'full_text_word_count']].agg(['mean', 'median', 'std'])
length_stats.index = ['real', 'fake']
length_stats.round(2)


## 7. Gráficos exploratorios

In [ ]:

# Distribución de clases
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='label', hue='label', palette=['#2ecc71', '#e74c3c'], legend=False, ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Real (0)', 'Fake (1)'])
ax.set_title('Distribución de clases')
ax.set_ylabel('Cantidad')
save_figure(fig, RESULTS_FIGURES / 'eda_class_distribution.png')
plt.show()

# Distribución temporal
fig, ax = plt.subplots(figsize=(12, 4))
monthly_plot = df.groupby(df['parsed_date'].dt.to_period('M')).size()
monthly_plot.index = monthly_plot.index.astype(str)
monthly_plot.plot(ax=ax)
ax.set_title('Cantidad de noticias por mes')
ax.set_xlabel('Mes')
ax.set_ylabel('Cantidad')
plt.xticks(rotation=45)
save_figure(fig, RESULTS_FIGURES / 'eda_temporal_distribution.png')
plt.show()


In [ ]:

# Longitud de títulos y cuerpos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ['title_word_count', 'text_word_count'], ['Títulos', 'Cuerpos']):
    sns.histplot(data=df, x=col, hue='label', bins=50, element='step', stat='density', common_norm=False, ax=ax)
    ax.set_title(f'Distribución de longitud — {title}')
    ax.legend(['Real', 'Fake'])
save_figure(fig, RESULTS_FIGURES / 'eda_text_length_distribution.png')
plt.show()


## 8. Análisis de `subject`

La columna `subject` **no se usará como feature** para entrenar modelos. Solo sirve para análisis exploratorio y para definir el subconjunto político.

**Advertencia:** `subject` puede funcionar como predictor demasiado fuerte y sesgado, ya que la distribución temática difiere mucho entre fake y real.

In [ ]:

subject_class = pd.crosstab(df['subject'], df['label'], margins=True)
subject_class.columns = ['real', 'fake', 'total']
subject_class['pct_fake'] = (subject_class['fake'] / subject_class['total'] * 100).round(1)
subject_class.sort_values('total', ascending=False)


In [ ]:

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = df.groupby(['subject', 'label']).size().reset_index(name='count')
sns.barplot(data=plot_df, y='subject', x='count', hue='label', ax=ax)
ax.set_title('Distribución por subject y clase')
ax.legend(['Real', 'Fake'])
save_figure(fig, RESULTS_FIGURES / 'eda_subject_distribution.png')
plt.show()

print('\n--- Conteos por subject y clase (para definir subconjunto político) ---')
politics_related = ['politicsNews', 'politics', 'Government News', 'left-news']
for subj in politics_related:
    sub = df[df['subject'] == subj]
    if len(sub) > 0:
        real_c = (sub['label'] == 0).sum()
        fake_c = (sub['label'] == 1).sum()
        print(f'{subj}: real={real_c:,}, fake={fake_c:,}, total={len(sub):,}')


### Decisión del subconjunto político

- **Reales:** `politicsNews` (único subject político en noticias reales)
- **Falsas:** `politics` (principal subject político en fake news)
- `Government News` y `left-news` son opcionales; se evalúan arriba pero el experimento principal usa solo `politics` para mantener comparabilidad temática.

## 9. Palabras frecuentes por clase

In [ ]:

from collections import Counter
import re

def top_words(texts, n=20):
    words = []
    for t in texts:
        tokens = re.findall(r'\b[a-z]{3,}\b', str(t).lower())
        words.extend(tokens)
    return Counter(words).most_common(n)

fake_words = top_words(df[df['label'] == 1]['full_text'])
real_words = top_words(df[df['label'] == 0]['full_text'])

print('Top términos FAKE:')
for w, c in fake_words:
    print(f'  {w}: {c:,}')
print('\nTop términos REAL:')
for w, c in real_words:
    print(f'  {w}: {c:,}')


In [ ]:

# Word clouds opcionales
try:
    from wordcloud import WordCloud

    for label, name, color in [(1, 'fake', '#e74c3c'), (0, 'real', '#2ecc71')]:
        text = ' '.join(df[df['label'] == label]['full_text'].astype(str))
        wc = WordCloud(width=800, height=400, background_color='white', colormap='Reds' if label else 'Greens').generate(text)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'Word cloud — {name}')
        save_figure(fig, RESULTS_FIGURES / f'eda_wordcloud_{name}.png')
        plt.show()
except ImportError:
    print('wordcloud no instalado; se omite word cloud')


## 10. Conclusiones preliminares del EDA

1. El dataset está relativamente balanceado entre fake y real.
2. Hay **fuerte asimetría temática**: las reales concentran `politicsNews` y `worldnews`; las falsas tienen múltiples subjects (`News`, `politics`, `left-news`, etc.).
3. Esto justifica el **experimento principal sobre el subconjunto político** para reducir sesgo temático.
4. Las noticias reales (Reuters) pueden tener marcas de estilo periodístico formal que el modelo podría aprender como proxy de veracidad.
5. El modelo posterior aprenderá **patrones del dataset**, no verificará hechos.

> Los duplicados detectados aquí se eliminan en el **Notebook 02** (criterio: `title` + `text`) antes del split temporal.